# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [19]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [20]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
import inspect

In [21]:
import os
import json
import importlib.util
import sys
import chromadb
import inspect
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
from typing import TypedDict
from lib.state_machine import StateMachine, Step, EntryPoint, Termination

In [22]:
print(inspect.signature(LLM.invoke))

(self, input: Union[str, lib.messages.BaseMessage, List[lib.messages.BaseMessage]], response_format: pydantic.main.BaseModel = None) -> lib.messages.AIMessage


In [23]:
from dotenv import load_dotenv
import os
 
load_dotenv("config.env", override=True)
 
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Key length:", len(os.getenv("OPENAI_API_KEY", "")))
print("Base URL:", os.getenv("OPENAI_BASE_URL"))

API key loaded: True
Key length: 49
Base URL: https://openai.vocareum.com/v1


In [24]:
## Stateful agent to manage the conversation and tools usage
from lib.agents import Agent
import os
 
# Create agent using the model available in config.env
conversation_agent = Agent(
    model_name="gpt-4o-mini",
    instructions="""
    You are a video game research assistant.
    Remember information provided earlier in the same session.
    Answer questions using the previous conversation when relevant.
    """
)
 
# Same session ID for all questions
session_id = "udaplay-demo-session"
 
print("QUESTION 1")
run1 = conversation_agent.invoke(
    "The game I am interested in is FIFA 21. Remember this game.",
    session_id=session_id
)
print(run1)
 
print("\nQUESTION 2")
run2 = conversation_agent.invoke(
    "What game did I say I was interested in?",
    session_id=session_id
)
print(run2)
 
print("\nQUESTION 3")
run3 = conversation_agent.invoke(
    "What was my previous question?",
    session_id=session_id
)
print(run3)

# Verify the actual answers returned by the agent
# Display the agent responses clearly
def get_answer(run):
    # Search snapshots from last to first
    for snapshot in reversed(run.snapshots):
        state = snapshot.state_data
 
        if "messages" in state:
            messages = state["messages"]
 
            # Find the latest assistant response
            for message in reversed(messages):
                if getattr(message, "role", None) == "assistant":
                    return message.content
 
    return "No assistant response found"
 
 
print("\n--- ANSWER 1 ---")
print(get_answer(run1))
 
print("\n--- ANSWER 2 ---")
print(get_answer(run2))
 
print("\n--- ANSWER 3 ---")
print(get_answer(run3))

QUESTION 1
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Run('fea443e3-5439-42be-9a2e-ab884c93199e')

QUESTION 2
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Run('6207058b-dcb6-4b91-8ab5-6bb573e875bf')

QUESTION 3
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Run('ac77a628-48fc-443e-b0c1-6bd574bed372')

--- ANSWER 1 ---
Got it! You're interested in FIFA 21. What would you like to know about it?

--- ANSWER 2 ---
You mentioned that you are interested in FIFA 21.

--- ANSWER 3 ---
Your previous question was about what game you said you were interested in, to which I responded that you mentioned FIFA 21.


In [25]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# Connect to the same persistent Chroma database
chroma_client = chromadb.PersistentClient(path="chroma_db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [26]:
print(chroma_client.list_collections())
 

[Collection(name=games)]


In [27]:
print("Embedding function ready")

Embedding function ready


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [28]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

def retrieve_game(query: str):
    # Get existing games collection
    collection = chroma_client.get_collection(name="games")
 
    # Semantic search
    results = collection.query(
        query_texts=[query],
        n_results=10,
        include=["documents", "metadatas", "distances"]
    )
 
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]
 
    retrieved_games = []
 
    # Prefer a game whose name/title appears directly in the question
    query_lower = query.lower()
 
    for document, metadata, distance in zip(
        documents, metadatas, distances
    ):
        game_name = (
            metadata.get("name")
            or metadata.get("Name")
            or metadata.get("title")
            or metadata.get("Title")
            or ""
        )
 
        if game_name and game_name.lower() in query_lower:
            retrieved_games.insert(
                0,
                {
                    "document": document,
                    "metadata": metadata,
                    "distance": distance
                }
            )
        else:
            retrieved_games.append(
                {
                    "document": document,
                    "metadata": metadata,
                    "distance": distance
                }
            )
 
    return retrieved_games[:3]

In [29]:
docs = retrieve_game("Who published FIFA 21?")
 
for doc in docs:
    print(doc)
    print()
 

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{'document': 'Platform: Xbox Series X|S\nName: Halo Infinite\nGenre: First-person shooter\nPublisher: Xbox Game Studios\nYearOfRelease: 2021\nSummary: ', 'metadata': {'platform': 'Xbox Series X|S', 'name': 'Halo Infinite'}, 'distance': 1.2950456142425537}

{'document': 'Platform: PlayStation 3\nName: Gran Turismo 5\nGenre: Racing\nPublisher: Sony Computer Entertainment\nYearOfRelease: 2010\nSummary: ', 'metadata': {'name': 'Gran Turismo 5', 'platform': 'PlayStation 3'}, 'distance': 1.2975733280181885}

{'document': 'Platform: Xbox One\nName: Minecraft\nGenre: Sandbox, Survival\nPublisher: Mojang Studios\nYearOfRelease: 2014\nSummary: ', 'metadata': {'name': 'Minecraft', 'platform': 'Xbox One'}, 'distance': 1.3063021898269653}



In [30]:
import os
from openai import OpenAI
 
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

print("Client created successfully")

Client created successfully


In [31]:
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))
print("NAVIGATE_API_KEY exists:", bool(os.getenv("NAVIGATE_API_KEY")))
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))

OPENAI_API_KEY exists: True
NAVIGATE_API_KEY exists: False
OPENAI_BASE_URL: https://openai.vocareum.com/v1


In [32]:
# Create LLM instance
llm = LLM()
from lib.messages import UserMessage
 
response = llm.invoke([
    UserMessage(content="Say hello")
])
 
print(response)

role='assistant' content='Hello! How can I assist you today?' tool_calls=None token_usage=TokenUsage(prompt_tokens=9, completion_tokens=9, total_tokens=18)


#### Evaluate Retrieval Tool

In [33]:
# Create LLM instance
llm = LLM()
# Evaluate Retrieval

def evaluate_retrieval(question, retrieved_docs):
 
    # No documents retrieved -> use web
    if not retrieved_docs:
        return {
            "result": "WEB",
            "description": "No relevant documents were retrieved"
        }
 
    # If the closest document has a good similarity distance,
    # use the Vector DB result
    best_distance = retrieved_docs[0].get("distance")
 
    if best_distance is not None and best_distance < 0.8:
        return {
            "result": "VECTOR",
            "description": "Relevant document found in Vector DB"
        }
 
    # Otherwise let the LLM evaluate the retrieved content
    system_prompt = """
You evaluate whether retrieved video-game documents contain
enough relevant information to answer the user's question.
 
Return exactly:
VECTOR
 
if the retrieved documents contain enough information to answer
the question.
 
Return exactly:
WEB
 
if the retrieved documents do not contain enough information.
 
Return only VECTOR or WEB.
"""
 
    user_prompt = f"""
Question:
{question}
 
Retrieved documents:
{retrieved_docs}
"""
 
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        UserMessage(content=user_prompt)
    ])
 
    decision = response.content.strip().upper()
 
    if decision == "VECTOR":
        return {
            "result": "VECTOR",
            "description": "LLM selected VECTOR route"
        }
 
    return {
        "result": "WEB",
        "description": "LLM selected WEB route"
    }

In [34]:
test_docs = [
    {
        "document": """
        FIFA 21 is a football simulation video game published by
        Electronic Arts. It was released in October 2020.
        """
    }
]
 
result = evaluate_retrieval(
    "Who published FIFA 21?",
    test_docs
)
 
print(result)

{'result': 'VECTOR', 'description': 'LLM selected VECTOR route'}


In [35]:
result = evaluate_retrieval(
    "Who developed FIFA 21?",
    test_docs
)
 
print(result)

{'result': 'WEB', 'description': 'LLM selected WEB route'}


In [36]:
question = "Who developed FIFA 21?"
 
retrieved_docs = retrieve_game(question)
 
evaluation = evaluate_retrieval(
    question,
    retrieved_docs
)
 
print(evaluation)

{'result': 'WEB', 'description': 'LLM selected WEB route'}


#### Game Web Search Tool

In [37]:
from tavily import TavilyClient
import os
 
def game_web_search(question):
    tavily_client = TavilyClient(
        api_key=os.getenv("TAVILY_API_KEY")
    )
 
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5
    )
 
    return response["results"]

In [38]:
web_results = game_web_search("Who developed FIFA 21?")
 
for result in web_results:
    print("Title:", result.get("title"))
    print("Content:", result.get("content"))
    print("URL:", result.get("url"))
    print("--------------------")

Title: FIFA 21
Content: Football simulation game

2020 video game

| FIFA 21 |

| Current Gen Standard cover art featuring Paris Saint-Germain player Kylian Mbappé |
| Developers | EA Vancouver EA Romania |
| Publisher | EA Sports |
| Series | FIFA "FIFA (video game series)") |
| Engine | Frostbite 3 "Frostbite (game engine)") |
| Platforms |  Microsoft Windows  Nintendo Switch  PlayStation 4  Xbox One  Stadia  PlayStation 5  Xbox Series X/S |
| Release |  Microsoft Windows, Nintendo Switch, PS4, Xbox One  9 October 2020  PS5, Xbox Series X/S  3 December 2020  Stadia  17 March 2021 |
| Genre | Sports |
| Modes | Single-player, multiplayer | [...] FIFA 21 is an association football simulation video game published by Electronic Arts as part of the FIFA series "FIFA (video game series)"). It is the 28th installment in the FIFA series, and was released on 9 October 2020 for Microsoft Windows, Nintendo Switch, PlayStation 4, and Xbox One. Enhanced versions for the PlayStation 5 and Xbox Ser

In [39]:
question = "Who developed FIFA 21?"
retrieved_docs = retrieve_game(question)
 
evaluation = evaluate_retrieval(
    question,
    retrieved_docs
)
 
print(evaluation)
print(type(evaluation))

{'result': 'WEB', 'description': 'LLM selected WEB route'}
<class 'dict'>


### Agent

In [40]:
class AgentState(TypedDict):
    question: str
    retrieved_docs: list
    evaluation: object
    web_results: list
    answer: str
 
 
workflow = StateMachine(AgentState)
 
 
# Step 1 - Retrieve from Vector DB
def retrieve_step(state: AgentState):
    docs = retrieve_game(state["question"])
 
    return {
        "retrieved_docs": docs
    }
 
def evaluate_step(state: AgentState):
    print("\nQUESTION:")
    print(state["question"])
 
    print("\nRETRIEVED DOCS:")
    print(state["retrieved_docs"])
 
    evaluation = evaluate_retrieval(
        state["question"],
        state["retrieved_docs"]
    )
 
    print("\nLLM EVALUATION:")
    print(evaluation)
 
    return {
        "evaluation": evaluation
    }
 
# Step 3 - Answer using Vector DB results
def vector_answer_step(state: AgentState):
 
    docs = state["retrieved_docs"]
 
    # Extract only the document text
    document_texts = []
 
    for doc in docs:
        if isinstance(doc, dict):
            document_texts.append(doc.get("document", ""))
        else:
            document_texts.append(str(doc))
 
    answer = "\n".join(document_texts)
 
    # Reviewer-required output
    print("\nTOOL USED: Vector Database")
    print("ROUTE: VECTOR")
    print("FINAL ANSWER:")
    print(answer)
    print("SOURCE: Internal Vector Database")
 
    return {
        "answer": answer
    }
 
 
# Step 4 - Search web if Vector DB was not useful
 
def web_search_step(state: AgentState):
 
    results = game_web_search(
        state["question"]
    )
 
    answer = str(results)
 
    # Reviewer-required output
    print("\nTOOL USED: Web Search")
    print("ROUTE: WEB")
    print("FINAL ANSWER:")
    print(answer)
 
    # Show web source/citation returned by the search tool
    print("SOURCE / WEBSITE:")
    print(results)
 
    return {
        "web_results": results,
        "answer": answer
    }
# Create workflow components
 
entry = EntryPoint()
 
retrieve_node = Step(
    "retrieve_game",
    retrieve_step
)
 
evaluate_node = Step(
    "evaluate_retrieval",
    evaluate_step
)
 
vector_answer_node = Step(
    "vector_answer",
    vector_answer_step
)
 
web_search_node = Step(
    "web_search",
    web_search_step
)
 
termination = Termination()
 
 
workflow.add_steps([
    entry,
    retrieve_node,
    evaluate_node,
    vector_answer_node,
    web_search_node,
    termination
])
 
 
workflow.connect(
    entry,
    retrieve_node
)
 
workflow.connect(
    retrieve_node,
    evaluate_node
)
 

In [41]:
# Router
def route_after_evaluation(state: AgentState):
    decision = state["evaluation"]["result"]
 
    if decision == "VECTOR":
        return [vector_answer_node]
 
    return [web_search_node]

In [42]:
# Connections
workflow.connect(
    evaluate_node,
    [vector_answer_node, web_search_node],
    route_after_evaluation
)
 
workflow.connect(
    vector_answer_node,
    termination
)
 
workflow.connect(
    web_search_node,
    termination
)

In [43]:
# Test 1 - question whose answer exists in your vector DB
initial_state = {
    "question": "What is Halo Infinite about?",
    "retrieved_docs": [],
    "evaluation": None,
    "web_results": [],
    "answer": ""
}
 
run_object = workflow.run(initial_state)
print(run_object)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve_game

QUESTION:
What is Halo Infinite about?

RETRIEVED DOCS:
[{'document': 'Platform: Xbox Series X|S\nName: Halo Infinite\nGenre: First-person shooter\nPublisher: Xbox Game Studios\nYearOfRelease: 2021\nSummary: ', 'metadata': {'name': 'Halo Infinite', 'platform': 'Xbox Series X|S'}, 'distance': 0.5476003289222717}, {'document': 'Platform: Xbox One\nName: Minecraft\nGenre: Sandbox, Survival\nPublisher: Mojang Studios\nYearOfRelease: 2014\nSummary: ', 'metadata': {'name': 'Minecraft', 'platform': 'Xbox One'}, 'distance': 1.3425931930541992}, {'document': 'Platform: Xbox 360\nName: Kinect Adventures!\nGenre: Party\nPublisher: Microsoft Game Studios\nYearOfRelease: 2010\nSummary: ', 'metadata': {'name': 'Kinect Adventures!', 'platform': 'Xbox 360'}, 'distance': 1.43696928024292}]

LLM EVALUATION:
{'result': 'VECTOR', 'description': 'Relevant document found in Vector DB'}
[StateMachine] Executing step: evaluate_re

In [44]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
# Test Agent
initial_state = {
    "question": "When Pokemon Gold and Silver was released?",
    "retrieved_docs": [],
    "evaluation": None,
    "web_results": [],
    "answer": ""
}
 
run_object = workflow.run(initial_state)
 
print(run_object)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve_game

QUESTION:
When Pokemon Gold and Silver was released?

RETRIEVED DOCS:
[{'document': 'Platform: Game Boy Color\nName: Pokémon Gold and Silver\nGenre: Role-playing\nPublisher: Nintendo\nYearOfRelease: 1999\nSummary: ', 'metadata': {'name': 'Pokémon Gold and Silver', 'platform': 'Game Boy Color'}, 'distance': 0.6845746040344238}, {'document': 'Platform: Game Boy Advance\nName: Pokémon Ruby and Sapphire\nGenre: Role-playing\nPublisher: Nintendo\nYearOfRelease: 2002\nSummary: ', 'metadata': {'name': 'Pokémon Ruby and Sapphire', 'platform': 'Game Boy Advance'}, 'distance': 0.8891581892967224}, {'document': 'Platform: Wii\nName: Wii Sports\nGenre: Sports\nPublisher: Nintendo\nYearOfRelease: 2006\nSummary: ', 'metadata': {'platform': 'Wii', 'name': 'Wii Sports'}, 'distance': 1.318616509437561}]

LLM EVALUATION:
{'result': 'VECTOR', 'description': 'Relevant document found in Vector DB'}
[StateMachine] Executing ste

In [45]:
# Demonstrate the agent using the three required example queries
 
test_queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]
 
for i, question in enumerate(test_queries, 1):
 
    print("\n" + "=" * 80)
    print(f"QUERY {i}: {question}")
    print("=" * 80)
 
    initial_state = {
        "question": question,
        "retrieved_docs": [],
        "evaluation": None,
        "web_results": [],
        "answer": ""
    }
 
    run_object = workflow.run(initial_state)
 
    print("\nAgent run completed.")
    print("Run object:", run_object)


QUERY 1: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve_game

QUESTION:
When Pokémon Gold and Silver was released?

RETRIEVED DOCS:
[{'document': 'Platform: Game Boy Color\nName: Pokémon Gold and Silver\nGenre: Role-playing\nPublisher: Nintendo\nYearOfRelease: 1999\nSummary: ', 'metadata': {'platform': 'Game Boy Color', 'name': 'Pokémon Gold and Silver'}, 'distance': 0.6551632881164551}, {'document': 'Platform: Game Boy Advance\nName: Pokémon Ruby and Sapphire\nGenre: Role-playing\nPublisher: Nintendo\nYearOfRelease: 2002\nSummary: ', 'metadata': {'name': 'Pokémon Ruby and Sapphire', 'platform': 'Game Boy Advance'}, 'distance': 0.9007943272590637}, {'document': 'Platform: Wii\nName: Wii Sports\nGenre: Sports\nPublisher: Nintendo\nYearOfRelease: 2006\nSummary: ', 'metadata': {'name': 'Wii Sports', 'platform': 'Wii'}, 'distance': 1.3348960876464844}]

LLM EVALUATION:
{'result': 'VECTOR', 'description': 'Relevant docu

### (Optional) Advanced

In [46]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes